# Pathology-constrained two-stage virtual H&E

This experiment assigns cell/nuclear structure synthesis to Stage 1 and restricts Stage 2 to physical H/E colorization. The deployed path is `Unstain OD → cell-aware predicted H&E OD → H/E concentrations → RGB H&E`. Real H&E OD is used only as the Stage-1 training target; it is never passed to Stage 2.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from cyclegan_core import denormalize, seed_everything
from pathology_constrained_staining import (
    CellAwareStructureTrainer, PathologyColorizerTrainer,
    load_cell_aware_structure_generator, soft_nuclei_map,
)
from two_stage_virtual_staining import build_two_stage_dataloaders

## Configuration

Both stages are new architectures and use separate v1 directories. No old Stage-1 or color checkpoint is loaded. `background_mask_blur_kernel` is not present because every background mask is hard (no blur).

In [ ]:
TRAIN_STAGE1 = True
TRAIN_STAGE2 = True

data_params = {
    'seed': 42,
    'gpu_index': 1,
    'data_dir': Path('../../data/HnE_n_UNStaining/patch_dataset_mpp05_2048'),
    'image_ext': 'png',
    'image_max_count': 30000,
    'original_size': 2048,
    'source_mpp': 0.5,
    'target_mpp': 2.0,
    'input_size': 512,
    'batch_size': 2,
    'val_fraction': 0.10,
    'preload_images': True,
    'max_cache_gib': 64,
    'od_background_threshold': 0.98,
    'od_quantile': 0.995,
    'od_calibration_images': 256,
}

stage1_params = {
    'output_dir': Path('../../results/Unstain2HnE_pathology_v1/structure'),
    'checkpoint_dir': Path('../../model/Unstain2HnE_pathology_v1/structure'),
    'num_epochs': 150,
    'base_channels': 32,
    'attention_blocks': 2,
    'attention_heads': 8,
    'attention_dropout': 0.10,
    'reverse_ngf': 32,
    'reverse_residual_blocks': 6,
    'ndf': 32,
    'lr_g': 2e-4,
    'lr_d': 1e-4,
    'beta1': 0.5,
    'beta2': 0.999,
    'weight_decay': 1e-4,
    'grad_clip': 5.0,
    'lambda_gan': 1.0,
    'lambda_cycle': 3.0,
    'lambda_identity': 1.0,
    'lambda_paired': 10.0,
    'lambda_ssim': 2.0,
    'lambda_gradient': 2.0,
    'lambda_laplacian': 1.0,
    'lambda_nuclei': 5.0,
    'lambda_cell_density': 2.0,
    'lambda_uncertainty': 0.25,
    'lambda_uncertainty_calibration': 0.50,
    'lambda_background': 5.0,
    'registration_shift_radius': 4,
    'registration_shift_step': 2,
    'nucleus_od_threshold': 0.50,
    'nucleus_temperature': 0.08,
    'cell_density_grid': 16,
    'a_background_od_threshold': 0.04,
    'b_background_od_threshold': 0.04,
    'pool_size': 50,
    'preview_count': 2,
    'save_every': 10,
}

stage2_params = {
    'output_dir': Path('../../results/Unstain2HnE_pathology_v1/color'),
    'checkpoint_dir': Path('../../model/Unstain2HnE_pathology_v1/color'),
    'num_epochs': 150,
    'base_channels': 32,
    'attention_blocks': 2,
    'attention_heads': 8,
    'attention_dropout': 0.10,
    'max_concentration': 4.0,
    'stain_basis_delta': 0.10,
    'ndf': 64,
    'lr_g': 2e-4,
    'lr_d': 1e-4,
    'beta1': 0.5,
    'beta2': 0.999,
    'weight_decay': 1e-4,
    'grad_clip': 5.0,
    'lambda_gan': 1.0,
    'lambda_rgb': 10.0,
    'lambda_ssim': 2.0,
    'lambda_gradient': 2.0,
    'lambda_laplacian': 1.0,
    'lambda_h': 3.0,
    'lambda_e': 2.0,
    'lambda_stain_moments': 1.0,
    'lambda_h_morphology': 2.0,
    'lambda_uncertainty': 0.25,
    'lambda_uncertainty_calibration': 0.50,
    'lambda_background': 10.0,
    'lambda_stain_basis': 1.0,
    'background_od_threshold': 0.04,
    'preview_count': 2,
    'save_every': 10,
}

seed_everything(data_params['seed'])
if torch.cuda.is_available():
    device = torch.device(f"cuda:{data_params['gpu_index']}")
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device('cpu')
print('device:', device)

## Slide-wise paired data

OD calibration is global, not per image. Stage 1 receives `(Unstain OD, Real H&E OD)`. Stage 2 receives `(Unstain OD, Real H&E RGB)`; its H&E-OD input is always generated by the best Stage 1.

In [ ]:
data = build_two_stage_dataloaders(data_params)
print('Unstain OD_MAX:', data['unstain_od_max'])
print('H&E OD_MAX:', data['hne_od_max'])

In [ ]:
unstain_od, real_hne_od = next(iter(data['structure_train']))
nuclei_response = soft_nuclei_map(
    real_hne_od, stage1_params['nucleus_od_threshold'],
    stage1_params['nucleus_temperature'],
)
columns = min(2, len(unstain_od))
fig, axes = plt.subplots(3, columns, figsize=(4 * columns, 10), squeeze=False)
for index in range(columns):
    axes[0, index].imshow(denormalize(unstain_od[index, 0]), cmap='gray', vmin=0, vmax=1)
    axes[0, index].set_title('Unstain OD')
    axes[1, index].imshow(denormalize(real_hne_od[index, 0]), cmap='gray', vmin=0, vmax=1)
    axes[1, index].set_title('Real H&E OD')
    axes[2, index].imshow(nuclei_response[index, 0], cmap='magma', vmin=0, vmax=1)
    axes[2, index].set_title('Soft nuclei target')
    for row in range(3):
        axes[row, index].axis('off')
plt.tight_layout()

## Stage 1 — cell-aware structure completion

The forward generator combines a sharp U-Net decoder with axial global attention and predicts both H&E OD and uncertainty. Bidirectional cycle learning remains active. A small whole-patch shift is selected before paired loss so residual registration error does not encourage averaging. Nuclear response, local cell-density, gradient, and Laplacian losses explicitly assign cellular morphology to Stage 1.

In [ ]:
stage1_trainer = CellAwareStructureTrainer(
    stage1_params, data['structure_train'], data['structure_val'],
    data['unstain_od_max'], device,
)
if TRAIN_STAGE1:
    stage1_trainer.fit()

In [ ]:
best_stage1_path = stage1_params['checkpoint_dir'] / 'best.pt'
stage1_generator, best_stage1 = load_cell_aware_structure_generator(
    best_stage1_path, device
)
stage1_trainer.G_AB.load_state_dict(best_stage1['G_AB'])
stage1_trainer.G_BA.load_state_dict(best_stage1['G_BA'])
print('Loaded best Stage 1 epoch:', best_stage1['epoch'] + 1)

## Stage 2 — physical H/E colorization

Stage 2 receives only predicted H&E OD. It predicts non-negative hematoxylin/eosin concentration maps plus an uncertainty map. RGB is reconstructed through a constrained H/E optical-density basis, preventing unrestricted purple color averaging.

In [ ]:
stage2_trainer = PathologyColorizerTrainer(
    stage2_params, data['color_train'], data['color_val'],
    stage1_generator, device,
)
if TRAIN_STAGE2:
    stage2_trainer.fit()
best_stage2_path = stage2_params['checkpoint_dir'] / 'best.pt'
best_stage2 = torch.load(best_stage2_path, map_location=device, weights_only=False)
stage2_trainer.G.load_state_dict(best_stage2['G_color'])
stage2_trainer.D.load_state_dict(best_stage2['D_color'])
print('Loaded best Stage 2 epoch:', best_stage2['epoch'] + 1)

## Final validation

Model selection uses the deployed predicted-OD path only. In addition to SSIM/PSNR, inspect H/E concentration errors, nuclear morphology, cell-density error, and uncertainty–error correlation. A useful uncertainty map should be brighter where actual OD/RGB error is larger.

In [ ]:
stage1_metrics, stage1_preview = stage1_trainer.validate()
stage2_metrics, stage2_preview = stage2_trainer.validate()
print('Stage 1:', {key: round(value, 4) for key, value in stage1_metrics.items()})
print('Stage 2:', {key: round(value, 4) for key, value in stage2_metrics.items()})
stage1_trainer.save_preview(stage1_params['num_epochs'] - 1, stage1_preview)
stage2_trainer.save_preview(stage2_params['num_epochs'] - 1, stage2_preview)